# 1.配置环境并准备数据集
1. 请确保在右侧**Input**中已经添加本次作业所需要的数据集**深度学习与地理应用-2025秋-作业3-PM2.5浓度估计**（默认会自动添加，如未，则搜索添加即可）
2. 若以上方法无法添加数据，请手动上传所需数据（可在作业首页下载数据）
3. 若要使用GPU或TPU，需要手动在右侧**Session options**中指定**ACCELERATOR**，Kaggle为免费用户每周提供30个小时的GPU和20个小时的TPU免费时长，已经足够
4. 若要托管代码，需要在右侧**Schedule a notebook to run**中保存或者直接点击右上角**Save Version**按钮，Kaggle会后台托管代码自动运行
5. 详细介绍和其余操作参见课程资源网站

In [ ]:
import pandas                                       # 提供dataframe，用来处理csv数据
import torch                                        # pytorch，目前主流的深度学习框架
from torch.utils import data                        # 从torch.utils导入data模块，用来构建pytorch中的数据结构--->dataloader
from torch import nn                                # 从torch导入nn模块，用来提供基本的神经网络接口，如全连接层nn.Linear()
import numpy as np                                  # 导入numpy，并将其重命名为np，numpy是一个高效的用来处理矩阵的库
from tqdm import tqdm                               # tqdm是一个可视化代码进程的模块
import os                                           # os库用来处理磁盘读写过程
from sklearn.preprocessing import StandardScaler    # 这是一个用来数据标准化的模块
import csv                                          # 读写csv文件的库

normalize = StandardScaler()                        # 实例化StandardScaler

# 为了处理大批量数据，在目前主流的深度学习框架下，数据都是以矩阵的形式表示的
# 而对于CPU来讲，其对于矩阵数据处理效率欠佳，而GPU(显卡）则擅长处理矩阵，因此深度学习模型都会部署到GPU上
# 这段代码的意思是，判断目前的设备中是否有安装CUDA（GPU加速模块）。
# 如果有的话，我们就让模型部署到GPU上，如果没有，我们继续用CPU（速度会慢一点）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
train_path = '/kaggle/input/dl-geo-2025fall-homework3/train.csv'
val_path = '/kaggle/input/dl-geo-2025fall-homework3/val.csv'
test_path = '/kaggle/input/dl-geo-2025fall-homework3/test.csv'

# 2.构建数据流

In [ ]:
# 这段代码用来创建pytorch所需的基本数据结构，data.Dataset
class PM2_5Dataset(data.Dataset):
    
    def __init__(self, csv_path, mode="train"):
        super(PM2_5Dataset, self).__init__()
        self.csv_content = pandas.read_csv(csv_path)
        self.used_column = ["AOD", "ET", "BLH", "TEM", "NDVI", "SP", "RH", "DEM", "NTL", "PRE", "WS", "WD"] # TODO：这些特征都是有用的吗？修改这行代码，可以修改模型输入的特征
        self.target_column = "PM2.5"
        self.mode = mode
        self.dim = len(self.used_column)
        if mode in ["train", "val"]:
            self.input_data, self.target_data = self.process_csv_with_gt()

        else:
            self.input_data = self.process_csv_without_gt()

    def process_csv_with_gt(self):
        input_data = [list(self.csv_content[i]) for i in self.used_column]
        input_data = np.array(input_data)
        input_data = input_data.T
        input_data = normalize.fit_transform(input_data)
        target_data = np.array(self.csv_content[self.target_column])
        return input_data, target_data

    def process_csv_without_gt(self):
        input_data = [list(self.csv_content[i]) for i in self.used_column]
        input_data = np.array(input_data)
        input_data = input_data.T
        input_data = normalize.fit_transform(input_data)
        return input_data

    def __getitem__(self, index):
        if self.mode in ["train", "val"]:
            data, gt = self.input_data[index, :], self.target_data[index]
            data = torch.from_numpy(data)
            gt = torch.tensor(gt)
            return data, gt
        else:
            data = self.input_data[index, :]
            return torch.from_numpy(data)

    def __len__(self):
        return self.input_data.shape[0]

def prepare_dataset(csv_path, batch_size, mode):
    dataset = PM2_5Dataset(csv_path, mode)
    # 在pytorch中，dataset需要用dataloader封装
    if mode in ["train", "val"]:
        dataloader = data.DataLoader(dataset, batch_size, num_workers=0, pin_memory=True)
    else:
        dataloader = data.DataLoader(dataset, 1, num_workers=0, pin_memory=True, shuffle=False)
    return dataloader

# 3.构建模型

In [ ]:
# 这是一个基本的神经网络模型，包含三个全连接层，激活函数为Relu
class MyNeuralNet(nn.Module):
    # TODO：增加约束，如添加dropput，batchnorm等。增删全连接层数。修改神经元数量。修改连接方式等。
    def __init__(self, input_dim):
        super(MyNeuralNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, input):
        predict = self.net(input)
        return predict

# 4.编写训练、验证和测试等Pipeline代码

In [ ]:
def train(train_dataset, val_dataset, config, model):
    """训练代码"""
    n_epochs = config['n_epochs'] # 训练次数
    optimizer = getattr(torch.optim, config['optimizer'])(model.parameters(), **config['optim_hparas']) # 优化器
    model.train() # 设置模型为训练模式（梯度可以回传）
    pbar = tqdm(total=n_epochs, desc="Train Mode", unit="epoch")
    loss = nn.MSELoss() # 损失函数为MSE(均方误差)
    loss_log = {"train_loss": [], "val_loss": []}
    min_mse = 1e16
    for epoch in range(n_epochs):
        pbar.update()
        epoch_mse = 0.0
        for x, y in train_dataset:
            optimizer.zero_grad() # 将优化器里储存的梯度清零
            x, y = x.to(device).float(), y.to(device).float()
            y = y.unsqueeze(1)
            pred = model(x)
            mse_loss = loss(pred, y) # 计算损失
            mse_loss.backward() # 损失回传
            optimizer.step() # 梯度更新
            loss_log["train_loss"].append(mse_loss.detach().cpu().item())
            epoch_mse += mse_loss.detach().cpu().item()
        pbar.set_postfix(epoch=epoch + 1, mse=epoch_mse / len(train_dataset))
        val_mse = val(val_dataset, model)
        if val_mse < min_mse:
            min_mse = val_mse
            torch.save(model.state_dict(), config['save_path'])
            print(f"best model saved! val mse:{val_mse},epoch:{epoch + 1}")
        loss_log["val_loss"].append(val_mse)
    return min_mse, loss_log

def val(val_dataset, model):
    """验证代码"""
    model.eval() # 设置为验证模式（梯度不回传）
    total_loss = 0
    loss = nn.MSELoss()
    for x, y in val_dataset:
        x, y = x.to(device).float(), y.to(device).float()
        y = y.unsqueeze(1)
        with torch.no_grad():
            pred = model(x) # 预测结果
            mse_loss = loss(pred, y) # 计算损失
        total_loss += mse_loss.detach().cpu().item()
    total_loss = total_loss / len(val_dataset)
    return total_loss


def test(test_set, model):
    """测试代码"""
    model.eval() #
    preds = []
    for x in test_set:
        x = x.to(device).float()
        with torch.no_grad():
            pred = model(x)
            preds.append(pred.detach().cpu())
    return preds

In [ ]:
# 这部分不需要修改
def process_preds_to_csv(preds):
    with open("MyPred.csv", "w", newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['id', 'PM2.5'])
        for i, p in enumerate(preds):
            p = p.cpu().item()
            writer.writerow([i, p])

# 5.模型训练和评估

In [ ]:
os.makedirs('/kaggle/working/models', exist_ok=True)
# 模型超参数设置
config = {
    'n_epochs': 200,  # 训练次数
    'batch_size': 128,  # mini-batch 的大小
    'optimizer': 'SGD',  # 优化算法 (optimizer in torch.optim)
    'optim_hparas': {  # 优化器的超参数(取决于你用了什么优化算法)
        'lr': 0.001,
        'momentum':0.9                # SGD的学习率  # SGD的动量
    },
    'save_path': 'models/model.pth'  # 存储路径
}
# 构建dataset
train_dataset = prepare_dataset(train_path, config["batch_size"], mode="train")
val_dataset = prepare_dataset(val_path, config["batch_size"], mode="val")
test_dataset = prepare_dataset(test_path, config["batch_size"], mode="test")
# 实例化模型
model = MyNeuralNet(train_dataset.dataset.dim).to(device)
# 开始训练与验证
min_mse, loss_log = train(train_dataset, val_dataset, config, model)

In [ ]:
import matplotlib.pyplot as plt # 用来绘制曲线
from matplotlib.pyplot import figure

def plot_learning_curve(loss_record, title=''):
    ''' 绘制损失函数曲线 '''
    total_steps = len(loss_record['train_loss'])
    x_1 = range(total_steps)
    x_2 = x_1[::len(loss_record['train_loss']) // len(loss_record['val_loss'])]
    figure(figsize=(6, 4))
    plt.plot(x_1, loss_record['train_loss'], c='tab:red', label='train')
    plt.plot(x_2, loss_record['val_loss'], c='tab:cyan', label='val')
    plt.ylim(0.0, 1000.)
    plt.xlabel('Training steps')
    plt.ylabel('MSE loss')
    plt.title('Learning curve of {}'.format(title))
    plt.legend()
    plt.show()

def plot_pred(dv_set, model, device, lim=35., title="", preds=None, targets=None):
    ''' 绘制你的预测结果和真实结果之间的分布情况 '''
    if preds is None or targets is None:
        model.eval()
        preds, targets = [], []
        for x, y in dv_set:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                pred = model(x.float())
                preds.append(pred.detach().cpu())
                targets.append(y.detach().cpu())
        preds = torch.cat(preds, dim=0).numpy()
        targets = torch.cat(targets, dim=0).numpy()

    figure(figsize=(5, 5))
    plt.scatter(targets, preds, c='r', alpha=0.5)
    plt.plot([-0.2, lim], [-0.2, lim], c='b')
    plt.xlim(-0.2, lim)
    plt.ylim(-0.2, lim)
    plt.xlabel('ground truth value')
    plt.ylabel('predicted value')
    plt.title(title)
    plt.show()


plot_learning_curve(loss_log,"DNN")

plot_pred(train_dataset,model,device,35., 'Ground Truth of Train v.s. Prediction')

plot_pred(val_dataset,model,device,35.,'Ground Truth of Val v.s. Prediction')

# 5.使用训练好的模型进行测试

In [ ]:
# 测试模型
preds = test(test_dataset, model)
# 将测试的结果保存到CSV文件里
process_preds_to_csv(preds)
# 执行完成后，可以从左侧下载预测结果(MyPred.csv)，并将其提交到Kaggle上，Kaggle会反馈给你测试集的评分